In [2]:
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
from tdc.single_pred import ADME

RDLogger.DisableLog("rdApp.*")

df = ADME(name="BBB_Martins").get_data()
df.to_csv("data/bbb_martins_raw.csv", index=False)
print(len(df), df.columns.tolist())
print(df["Y"].value_counts(normalize=True))

df["mol"] = df["Drug"].apply(Chem.MolFromSmiles)
print("unparseable:", df["mol"].isna().sum())
df = df[df["mol"].notna()]

df["ikey"] = df["mol"].apply(Chem.MolToInchiKey)
dupes = df[df.duplicated("ikey", keep=False)]
print("rows in duplicate groups:", len(dupes))
print("conflicting-label groups:",(dupes.groupby("ikey")["Y"].nunique() > 1).sum())

df["scaffold"] = df["mol"].apply(lambda m: MurckoScaffold.MurckoScaffoldSmiles(mol=m, includeChirality=False))
print("unique scaffolds:", df["scaffold"].nunique())
print("singleton scaffolds:", (df["scaffold"].value_counts() == 1).sum())
print("largest scaffold classes:\n", df["scaffold"].value_counts().head())

Downloading...
100%|██████████| 138k/138k [00:00<00:00, 872kiB/s] 
Loading...
Done!


2030 ['Drug_ID', 'Drug', 'Y']
Y
1    0.764039
0    0.235961
Name: proportion, dtype: float64
unparseable: 0
rows in duplicate groups: 115
conflicting-label groups: 11
unique scaffolds: 1025
singleton scaffolds: 764
largest scaffold classes:
 scaffold
c1ccccc1                           136
                                    99
O=C1C=CC2C(=C1)CCC1C3CCCC3CCC21     76
O=C1CN=C(c2ccccc2)c2ccccc2N1        26
c1ccc2c(c1)Nc1ccccc1S2              24
Name: count, dtype: int64
